In [1]:
# Part 6: Filtering & Selection
# AI-assisted in silico design of antibody variants targeting Influenza Hemagglutinin

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from Bio.SeqUtils.ProtParam import ProteinAnalysis
import warnings
warnings.filterwarnings('ignore')

print("Part 6: Filtering & Selection Started")
print("=" * 50)

# Load Top 3 candidates from Part 5
top_3_candidates = [
    {"variant": "YGSTGDRH", "mutation": "G1Y", "kd_nm": 205.8, "final_score": 1.037, "improvement": 5.0},
    {"variant": "FGSTGDRH", "mutation": "G1F", "kd_nm": 303.3, "final_score": 1.006, "improvement": 3.4},
    {"variant": "NGSTGDRH", "mutation": "G1N", "kd_nm": 308.5, "final_score": 0.803, "improvement": 3.4}
]

print(f"Evaluating {len(top_3_candidates)} final candidates")
print("Target: Select single best antibody variant")

Part 6: Filtering & Selection Started
Evaluating 3 final candidates
Target: Select single best antibody variant


In [2]:
# Multi-Criteria Evaluation Framework
def evaluate_drug_like_properties(candidates):
    """
    Evaluate ADMET (Absorption, Distribution, Metabolism, Excretion, Toxicity) properties
    """
    
    evaluation_results = []
    
    for candidate in candidates:
        variant = candidate['variant']
        sequence = variant
        
        # Physicochemical analysis
        analysis = ProteinAnalysis(sequence)
        
        # Calculate properties
        molecular_weight = analysis.molecular_weight()
        hydrophobicity = analysis.gravy()  # Grand average of hydropathy
        instability_index = analysis.instability_index()
        flexibility = analysis.flexibility()
        secondary_structure = analysis.secondary_structure_fraction()
        
        # Drug-like property scoring (0-1 scale)
        
        # 1. Molecular Weight Score (optimal: 800-900 Da for peptides)
        mw_score = 1.0 if 800 <= molecular_weight <= 900 else 0.8
        
        # 2. Hydrophobicity Score (optimal: -0.5 to 0.5 for peptides)
        hydro_score = 1.0 if -0.5 <= hydrophobicity <= 0.5 else 0.7
        
        # 3. Stability Score (lower instability index = more stable)
        stability_score = 1.0 if instability_index < 40 else 0.8
        
        # 4. Flexibility Score (moderate flexibility good for binding)
        avg_flexibility = np.mean(flexibility) if flexibility else 1.0
        flex_score = 1.0 if 0.9 <= avg_flexibility <= 1.1 else 0.8
        
        # 5. Secondary Structure Score (balanced structure preferred)
        helix_fraction = secondary_structure[0]
        struct_score = 0.9 if helix_fraction < 0.3 else 0.7  # Less helix better for loops
        
        # Calculate overall drug-like score
        drug_like_score = np.mean([mw_score, hydro_score, stability_score, flex_score, struct_score])
        
        evaluation_results.append({
            'variant': variant,
            'mutation': candidate['mutation'],
            'kd_nm': candidate['kd_nm'],
            'molecular_weight': round(molecular_weight, 1),
            'hydrophobicity': round(hydrophobicity, 3),
            'instability_index': round(instability_index, 1),
            'avg_flexibility': round(avg_flexibility, 3),
            'helix_fraction': round(helix_fraction, 3),
            'mw_score': mw_score,
            'hydro_score': hydro_score,
            'stability_score': stability_score,
            'flex_score': flex_score,
            'struct_score': struct_score,
            'drug_like_score': round(drug_like_score, 3)
        })
    
    return evaluation_results

# Evaluate drug-like properties
print("🔬 MULTI-CRITERIA EVALUATION:")
print("=" * 70)

drug_evaluation = evaluate_drug_like_properties(top_3_candidates)

# Display results
print(f"{'Variant':<10} {'MW (Da)':<8} {'Hydro':<7} {'Stab':<6} {'Flex':<6} {'Drug Score':<10}")
print("-" * 70)

for result in drug_evaluation:
    print(f"{result['variant']:<10} {result['molecular_weight']:<8} {result['hydrophobicity']:<7} "
          f"{result['instability_index']:<6} {result['avg_flexibility']:<6} {result['drug_like_score']:<10}")

print("\n DETAILED SCORING BREAKDOWN:")
print("-" * 70)
for result in drug_evaluation:
    print(f"\n{result['variant']} ({result['mutation']}):")
    print(f"  MW Score: {result['mw_score']:.1f} | Hydro: {result['hydro_score']:.1f} | "
          f"Stab: {result['stability_score']:.1f} | Flex: {result['flex_score']:.1f} | "
          f"Struct: {result['struct_score']:.1f}")
    print(f"  Overall Drug-like Score: {result['drug_like_score']:.3f}")

🔬 MULTI-CRITERIA EVALUATION:
Variant    MW (Da)  Hydro   Stab   Flex   Drug Score
----------------------------------------------------------------------
YGSTGDRH   891.9    -1.85   2.2    1.0    0.92      
FGSTGDRH   875.9    -1.338  12.8   1.0    0.92      
NGSTGDRH   842.8    -2.125  -6.0   1.0    0.92      

📊 DETAILED SCORING BREAKDOWN:
----------------------------------------------------------------------

YGSTGDRH (G1Y):
  MW Score: 1.0 | Hydro: 0.7 | Stab: 1.0 | Flex: 1.0 | Struct: 0.9
  Overall Drug-like Score: 0.920

FGSTGDRH (G1F):
  MW Score: 1.0 | Hydro: 0.7 | Stab: 1.0 | Flex: 1.0 | Struct: 0.9
  Overall Drug-like Score: 0.920

NGSTGDRH (G1N):
  MW Score: 1.0 | Hydro: 0.7 | Stab: 1.0 | Flex: 1.0 | Struct: 0.9
  Overall Drug-like Score: 0.920


In [3]:
# Risk Assessment & Manufacturability Analysis
def assess_risks_and_manufacturability(drug_evaluation, top_3_candidates):
    """
    Evaluate manufacturing feasibility and potential risks
    """
    
    risk_analysis = []
    
    for i, result in enumerate(drug_evaluation):
        variant = result['variant']
        candidate = top_3_candidates[i]
        
        # Manufacturing Risk Assessment
        manufacturing_risks = {
            'aggregation_risk': 0,
            'oxidation_risk': 0, 
            'deamidation_risk': 0,
            'expression_difficulty': 0
        }
        
        # Analyze sequence for manufacturing risks
        for j, aa in enumerate(variant):
            # Aggregation risk (hydrophobic residues)
            if aa in ['F', 'Y', 'W', 'I', 'L', 'V', 'M']:
                manufacturing_risks['aggregation_risk'] += 1
            
            # Oxidation risk (Met, Cys, Trp, Tyr)
            if aa in ['M', 'C', 'W', 'Y']:
                manufacturing_risks['oxidation_risk'] += 1
            
            # Deamidation risk (Asn, Gln)
            if aa in ['N', 'Q']:
                manufacturing_risks['deamidation_risk'] += 1
        
        # Expression difficulty (complex amino acids)
        complex_aa = sum([1 for aa in variant if aa in ['W', 'C', 'M', 'P']])
        manufacturing_risks['expression_difficulty'] = complex_aa
        
        # Calculate manufacturing score (lower risk = higher score)
        total_risk = sum(manufacturing_risks.values())
        manufacturing_score = max(0.5, 1.0 - (total_risk * 0.1))  # Min 0.5
        
        # Immunogenicity Risk (aromatic content)
        aromatic_count = sum([1 for aa in variant if aa in ['F', 'Y', 'W']])
        immuno_risk = "Low" if aromatic_count <= 1 else "Medium"
        immuno_score = 0.9 if aromatic_count <= 1 else 0.7
        
        # Clinical Potential Score
        # Based on binding improvement and drug-like properties
        binding_improvement = candidate['improvement']
        clinical_score = min(1.0, (binding_improvement / 5.0) * result['drug_like_score'])
        
        risk_analysis.append({
            'variant': variant,
            'mutation': result['mutation'],
            'kd_nm': result['kd_nm'],
            'aggregation_risk': manufacturing_risks['aggregation_risk'],
            'oxidation_risk': manufacturing_risks['oxidation_risk'],
            'deamidation_risk': manufacturing_risks['deamidation_risk'],
            'expression_difficulty': manufacturing_risks['expression_difficulty'],
            'manufacturing_score': round(manufacturing_score, 3),
            'immunogenicity_risk': immuno_risk,
            'immunogenicity_score': immuno_score,
            'clinical_potential': round(clinical_score, 3),
            'total_risk_factors': total_risk
        })
    
    return risk_analysis

# Perform risk assessment
print("\n  RISK ASSESSMENT & MANUFACTURABILITY:")
print("=" * 80)

risk_assessment = assess_risks_and_manufacturability(drug_evaluation, top_3_candidates)

print(f"{'Variant':<10} {'Agg':<4} {'Ox':<3} {'Deam':<5} {'Expr':<5} {'Manuf':<6} {'Immuno':<7} {'Clinical':<8}")
print("-" * 80)

for result in risk_assessment:
    print(f"{result['variant']:<10} {result['aggregation_risk']:<4} {result['oxidation_risk']:<3} "
          f"{result['deamidation_risk']:<5} {result['expression_difficulty']:<5} "
          f"{result['manufacturing_score']:<6} {result['immunogenicity_risk']:<7} {result['clinical_potential']:<8}")

print("\n RISK DETAILS:")
for result in risk_assessment:
    print(f"\n{result['variant']} ({result['mutation']}):")
    print(f"  Manufacturing Score: {result['manufacturing_score']} (higher = better)")
    print(f"  Immunogenicity: {result['immunogenicity_risk']} risk")
    print(f"  Clinical Potential: {result['clinical_potential']}")
    print(f"  Total Risk Factors: {result['total_risk_factors']}")


  RISK ASSESSMENT & MANUFACTURABILITY:
Variant    Agg  Ox  Deam  Expr  Manuf  Immuno  Clinical
--------------------------------------------------------------------------------
YGSTGDRH   1    1   0     0     0.8    Low     0.92    
FGSTGDRH   1    0   0     0     0.9    Low     0.626   
NGSTGDRH   0    0   1     0     0.9    Low     0.626   

 RISK DETAILS:

YGSTGDRH (G1Y):
  Manufacturing Score: 0.8 (higher = better)
  Immunogenicity: Low risk
  Clinical Potential: 0.92
  Total Risk Factors: 2

FGSTGDRH (G1F):
  Manufacturing Score: 0.9 (higher = better)
  Immunogenicity: Low risk
  Clinical Potential: 0.626
  Total Risk Factors: 1

NGSTGDRH (G1N):
  Manufacturing Score: 0.9 (higher = better)
  Immunogenicity: Low risk
  Clinical Potential: 0.626
  Total Risk Factors: 1


In [4]:
# Final Decision Matrix & Candidate Selection
def create_final_decision_matrix(top_3_candidates, drug_evaluation, risk_assessment):
    """
    Create comprehensive decision matrix with weighted criteria
    """
    
    decision_matrix = []
    
    # Define criteria weights (total = 1.0)
    weights = {
        'binding_affinity': 0.35,    # Most important
        'drug_like_properties': 0.20,
        'manufacturing': 0.15,
        'clinical_potential': 0.15,
        'immunogenicity': 0.10,
        'structural_confidence': 0.05
    }
    
    for i, candidate in enumerate(top_3_candidates):
        variant = candidate['variant']
        
        # Normalize binding affinity (lower Kd = higher score)
        kd_values = [c['kd_nm'] for c in top_3_candidates]
        min_kd = min(kd_values)
        max_kd = max(kd_values)
        binding_score = 1 - (candidate['kd_nm'] - min_kd) / (max_kd - min_kd) if max_kd != min_kd else 1.0
        
        # Get other scores
        drug_score = drug_evaluation[i]['drug_like_score']
        manuf_score = risk_assessment[i]['manufacturing_score']
        clinical_score = risk_assessment[i]['clinical_potential']
        immuno_score = risk_assessment[i]['immunogenicity_score']
        
        # Structural confidence (from Part 4 - simulate AlphaFold2 confidence)
        struct_confidence = 0.85 if variant == 'YGSTGDRH' else 0.82 if variant == 'FGSTGDRH' else 0.80
        
        # Calculate weighted final score
        final_score = (
            binding_score * weights['binding_affinity'] +
            drug_score * weights['drug_like_properties'] +
            manuf_score * weights['manufacturing'] +
            clinical_score * weights['clinical_potential'] +
            immuno_score * weights['immunogenicity'] +
            struct_confidence * weights['structural_confidence']
        )
        
        decision_matrix.append({
            'variant': variant,
            'mutation': candidate['mutation'],
            'kd_nm': candidate['kd_nm'],
            'binding_score': round(binding_score, 3),
            'drug_score': round(drug_score, 3),
            'manufacturing_score': round(manuf_score, 3),
            'clinical_score': round(clinical_score, 3),
            'immunogenicity_score': round(immuno_score, 3),
            'structural_score': round(struct_confidence, 3),
            'weighted_final_score': round(final_score, 3),
            'improvement_fold': candidate['improvement']
        })
    
    # Sort by final score
    decision_matrix.sort(key=lambda x: x['weighted_final_score'], reverse=True)
    
    return decision_matrix, weights

# Create final decision matrix
print("\n FINAL DECISION MATRIX:")
print("=" * 100)

final_matrix, criteria_weights = create_final_decision_matrix(top_3_candidates, drug_evaluation, risk_assessment)

print("Criteria Weights:")
for criterion, weight in criteria_weights.items():
    print(f"  {criterion.replace('_', ' ').title()}: {weight:.0%}")

print(f"\n{'Rank':<4} {'Variant':<10} {'Binding':<8} {'Drug':<6} {'Manuf':<7} {'Clinical':<8} "
      f"{'Immuno':<7} {'Struct':<7} {'Final Score':<11} {'Improvement':<11}")
print("-" * 100)

for rank, result in enumerate(final_matrix, 1):
    print(f"{rank:<4} {result['variant']:<10} {result['binding_score']:<8} {result['drug_score']:<6} "
          f"{result['manufacturing_score']:<7} {result['clinical_score']:<8} {result['immunogenicity_score']:<7} "
          f"{result['structural_score']:<7} {result['weighted_final_score']:<11} {result['improvement_fold']:<11}x")

print(f"\n FINAL WINNER: {final_matrix[0]['variant']} ({final_matrix[0]['mutation']})")
print(f"   Final Score: {final_matrix[0]['weighted_final_score']}")
print(f"   Binding Affinity: {final_matrix[0]['kd_nm']} nM")
print(f"   Improvement over Original: {final_matrix[0]['improvement_fold']}x")

print(f"\n WINNER PROFILE:")
winner = final_matrix[0]
print(f"   Sequence: {winner['variant']}")
print(f"   Mutation: {winner['mutation']} (G1Y substitution)")
print(f"   Binding: Excellent ({winner['kd_nm']} nM)")
print(f"   Manufacturing: Good (score: {winner['manufacturing_score']})")
print(f"   Clinical Potential: High (score: {winner['clinical_score']})")
print(f"   Overall Assessment: RECOMMENDED FOR DEVELOPMENT")


 FINAL DECISION MATRIX:
Criteria Weights:
  Binding Affinity: 35%
  Drug Like Properties: 20%
  Manufacturing: 15%
  Clinical Potential: 15%
  Immunogenicity: 10%
  Structural Confidence: 5%

Rank Variant    Binding  Drug   Manuf   Clinical Immuno  Struct  Final Score Improvement
----------------------------------------------------------------------------------------------------
1    YGSTGDRH   1.0      0.92   0.8     0.92     0.9     0.85    0.924       5.0        x
2    FGSTGDRH   0.051    0.92   0.9     0.626    0.9     0.82    0.562       3.4        x
3    NGSTGDRH   0.0      0.92   0.9     0.626    0.9     0.8     0.543       3.4        x

 FINAL WINNER: YGSTGDRH (G1Y)
   Final Score: 0.924
   Binding Affinity: 205.8 nM
   Improvement over Original: 5.0x

 WINNER PROFILE:
   Sequence: YGSTGDRH
   Mutation: G1Y (G1Y substitution)
   Binding: Excellent (205.8 nM)
   Manufacturing: Good (score: 0.8)
   Clinical Potential: High (score: 0.92)
   Overall Assessment: RECOMMENDED FOR DEV